In [ ]:
### ▶ Step 1 — Load model & environment
# Run Block 1 once per session.

# Notes:
# - This block provides ONE UI to load either LLaVA or Qwen2.5-VL.
# - We keep one set of global variables: model, processor, model_id, VLM_BACKEND
# - For 4bit/8bit: use device_map="auto" and DO NOT call model.to("cuda")

### ▶ Step 2 — Define prompt & scoring logic
# You can rerun Block 2 as many times as needed without reloading the model.

### ▶ Step 3 — Run scoring
# Execute Block 3 to process images

In [1]:
# ===============================================
# BLOCK 1 — Environment + VLM Loader (LLaVA + Qwen2.5-VL)
# ===============================================

!pip install --no-cache-dir -q git+https://github.com/huggingface/transformers accelerate bitsandbytes qwen-vl-utils

import time
import torch
import ipywidgets as widgets
from IPython.display import display
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
)

# ------------------------------------------------
# Available model registries
# ------------------------------------------------
LLAVA_MODELS = {
    "[LLaVA] LLaVA v1.6 Mistral 7B": ("llava", "llava-hf/llava-v1.6-mistral-7b-hf"),
    "[LLaVA] LLaVA v1.6 Vicuna 7B": ("llava", "llava-hf/llava-v1.6-vicuna-7b-hf"),
    "[LLaVA] LLaVA v1.6 Vicuna 13B": ("llava", "llava-hf/llava-v1.6-vicuna-13b-hf"),
    "[LLaVA] LLaVA v1.6 34B": ("llava", "llava-hf/llava-v1.6-34b-hf"),
    "[LLaVA] LLaMA3 LLaVA-NeXT 8B": ("llava", "llava-hf/llama3-llava-next-8b-hf"),
    "[LLaVA] LLaVA-NeXT 72B": ("llava", "llava-hf/llava-next-72b-hf"),
    "[LLaVA] LLaVA-NeXT 110B": ("llava", "llava-hf/llava-next-110b-hf"),
}

QWEN_MODELS = {
    "[Qwen]  Qwen2.5-VL 3B Instruct": ("qwen", "Qwen/Qwen2.5-VL-3B-Instruct"),
    "[Qwen]  Qwen2.5-VL 7B Instruct": ("qwen", "Qwen/Qwen2.5-VL-7B-Instruct"),
    "[Qwen]  Qwen2.5-VL 32B Instruct": ("qwen", "Qwen/Qwen2.5-VL-32B-Instruct"),
    "[Qwen]  Qwen2.5-VL 72B Instruct": ("qwen", "Qwen/Qwen2.5-VL-72B-Instruct"),
}

MODEL_CHOICES = {**LLAVA_MODELS, **QWEN_MODELS}

# ------------------------------------------------
# UI for model selection
# ------------------------------------------------
use_token_widget = widgets.Checkbox(
    value=True,
    description="Use Hugging Face token",
)

model_widget = widgets.Dropdown(
    options=list(MODEL_CHOICES.keys()),
    value="[Qwen]  Qwen2.5-VL 7B Instruct",
    description="Model",
)

precision_widget = widgets.Dropdown(
    options=[
        ("4-bit (bnb, low VRAM)", "4bit"),
        ("8-bit (bnb)", "8bit"),
        ("FP16 / auto", "fp16"),
    ],
    value="4bit",
    description="Precision",
)

low_cpu_widget = widgets.Checkbox(
    value=True,
    description="low_cpu_mem_usage",
)

device_map_widget = widgets.Dropdown(
    options=[
        ("GPU only (cuda:0)", "cuda0"),
        ("Auto (accelerate decides)", "auto"),
        ("GPU + CPU offload (auto + offload folder)", "offload"),
    ],
    value="auto",
    description="Placement",
)

load_button = widgets.Button(
    description="Load model",
    button_style="success",
)

status_html = widgets.HTML(value="")

# ------------------------------------------------
# Loader callback
# ------------------------------------------------
def load_model(b):
    global model, processor, model_id, VLM_BACKEND, HF_TOKEN
    global DEVICE_MAP_MODE, MODEL_MAIN_DEVICE
    global QWEN_MIN_PIXELS, QWEN_MAX_PIXELS
    global gpu_name

    status_html.value = "<b>⏳ Loading model...</b>"
    start = time.time()
    ok = False
    err_msg = ""

    try:
        # Auth
        HF_TOKEN = None
        auth_kwargs = {}
        if use_token_widget.value:
            try:
                HF_TOKEN = userdata.get("HF_TOKEN")
                login(token=HF_TOKEN)
                auth_kwargs["token"] = HF_TOKEN
            except Exception:
                pass

        # Parse selection
        VLM_BACKEND, model_id = MODEL_CHOICES[model_widget.value]
        precision = precision_widget.value
        low_cpu = low_cpu_widget.value
        DEVICE_MAP_MODE = device_map_widget.value

        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

        # Quantization config
        quantization_config = None
        if precision == "4bit":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
        elif precision == "8bit":
            quantization_config = BitsAndBytesConfig(load_in_8bit=True)

        # Device map (DO NOT include quantization_config here - pass it separately)
        load_kwargs_common = {}
        if DEVICE_MAP_MODE == "auto":
            load_kwargs_common["device_map"] = "auto"
        elif DEVICE_MAP_MODE == "offload":
            load_kwargs_common["device_map"] = "auto"
            load_kwargs_common["offload_folder"] = "/content/offload"
        # cuda0 handled separately

        # Default Qwen pixel settings
        QWEN_MIN_PIXELS = 256 * 28 * 28
        QWEN_MAX_PIXELS = 640 * 28 * 28

        # Load model based on backend
        if VLM_BACKEND == "llava":
            processor = LlavaNextProcessor.from_pretrained(model_id, **auth_kwargs)

            if quantization_config:
                model = LlavaNextForConditionalGeneration.from_pretrained(
                    model_id,
                    torch_dtype=torch.float16,
                    low_cpu_mem_usage=low_cpu,
                    quantization_config=quantization_config,
                    **load_kwargs_common,
                    **auth_kwargs
                )
            else:
                model = LlavaNextForConditionalGeneration.from_pretrained(
                    model_id,
                    torch_dtype=torch.float16,
                    low_cpu_mem_usage=low_cpu,
                    **load_kwargs_common,
                    **auth_kwargs
                )

                if DEVICE_MAP_MODE == "cuda0" and torch.cuda.is_available():
                    model = model.to("cuda")

        elif VLM_BACKEND == "qwen":
            processor = AutoProcessor.from_pretrained(
                model_id,
                min_pixels=QWEN_MIN_PIXELS,
                max_pixels=QWEN_MAX_PIXELS,
                **auth_kwargs
            )

            torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

            if quantization_config:
                model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                    model_id,
                    torch_dtype=torch_dtype,
                    quantization_config=quantization_config,
                    low_cpu_mem_usage=low_cpu,
                    **load_kwargs_common,
                    **auth_kwargs
                )
            else:
                model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                    model_id,
                    torch_dtype="auto" if precision == "fp16" else torch_dtype,
                    low_cpu_mem_usage=low_cpu,
                    **load_kwargs_common,
                    **auth_kwargs
                )
        else:
            raise ValueError(f"Unknown backend: {VLM_BACKEND}")

        print("• Device map:", getattr(model, "hf_device_map", "single-device"))

        if hasattr(model, "hf_device_map"):
            MODEL_MAIN_DEVICE = None
        else:
            MODEL_MAIN_DEVICE = model.device

        ok = True

    except Exception as e:
        err_msg = str(e)
        raise

    finally:
        elapsed_min = (time.time() - start) / 60

        model_label = next(
            k for k, v in MODEL_CHOICES.items()
            if v == (VLM_BACKEND, model_id)
        )

        if ok:
            status_html.value = (
                f"<b>✅ Model loaded</b><br>"
                f"⏱️ Time: <b>{elapsed_min:.2f}</b> minutes<br>"
                f"🔥 {model_label} on {gpu_name} ({precision})"
            )
        else:
            status_html.value = (
                f"<b>❌ Load failed</b><br>"
                f"⏱️ Time: <b>{elapsed_min:.2f}</b> minutes<br>"
                f"🔥 {model_label} on {gpu_name} ({precision})<br>"
                f"<code>{err_msg}</code>"
            )

load_button.on_click(load_model)

ui = widgets.VBox([
    use_token_widget,
    model_widget,
    precision_widget,
    low_cpu_widget,
    device_map_widget,
    load_button,
    status_html,
])

display(ui)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

• Device map: single-device


In [2]:
# ===============================================
# BLOCK 2 – Drive + Paths + Multi-Task Prompt Form + Forward Pass
#           With Task Type Support + Qwen Pixel Settings + Seed Toggle
#           + CONSENSUS VALIDATION + ADVANCED REASONING (v2.2.2)
#           FIX: numeric_tolerance greyed out for tasks 2+
#           FIX: consensus max runs increased to 5
#           FIX: Consensus now properly ignores NA values
# ===============================================

from google.colab import drive
drive.mount('/content/drive')

import os
import random
import numpy as np
import torch
import re
import requests
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display
from collections import Counter

from qwen_vl_utils import process_vision_info
from transformers import GenerationConfig, AutoProcessor

# ------------------------------------------------
# Task Type Definitions
# ------------------------------------------------
TASK_TYPES = {
    "numeric": "Extracts the last number from response (for counting, scores, ratings)",
    "category": "Returns the full text response stripped (for classifications)",
    "boolean": "Extracts yes/no/true/false and normalizes to 1/0",
    "text": "Returns raw response as-is (for descriptions, free-form)"
}

# ------------------------------------------------
# Advanced Reasoning Format Templates (NEW in v2.2)
# ------------------------------------------------
ADVANCED_REASONING_FORMATS = {
    "numeric": "First, describe what you observe and explain your reasoning step by step.\nThen, on the last line, write only: ANSWER: <integer>",
    "category": "First, describe what you observe and explain your reasoning step by step.\nThen, on the last line, write only: ANSWER: <category number>",
    "boolean": "First, describe what you observe and explain your reasoning step by step.\nThen, on the last line, write only: ANSWER: <yes or no>",
}

# Recommended minimum tokens for advanced reasoning (reference only, not auto-applied)
ADVANCED_REASONING_MAX_TOKENS = 1024

# ------------------------------------------------
# Response Parsing Functions by Task Type
# ------------------------------------------------
def parse_numeric(raw: str) -> str:
    """Extract the last number from the response."""
    numbers = re.findall(r"-?\d+(?:\.\d+)?", raw)
    return numbers[-1] if numbers else "NA"


def parse_category(raw: str) -> str:
    """
    Return cleaned text response for categorical outputs.
    Strips whitespace, removes common prefixes, and handles multi-line.
    """
    text = raw.strip()

    prefixes_to_remove = [
        "The answer is:",
        "Answer:",
        "Category:",
        "Classification:",
        "This is a",
        "This is an",
        "This appears to be a",
        "This appears to be an",
        "I would classify this as",
        "This can be classified as",
        "Based on the image,",
        "Looking at the image,",
    ]

    text_lower = text.lower()
    for prefix in prefixes_to_remove:
        if text_lower.startswith(prefix.lower()):
            text = text[len(prefix):].strip()
            text_lower = text.lower()

    text = text.rstrip('.')

    if '\n' in text:
        first_line = text.split('\n')[0].strip()
        if first_line and len(first_line) < 100:
            text = first_line

    return text if text else "NA"


def parse_boolean(raw: str) -> str:
    """
    Extract boolean response and normalize to 1/0.
    """
    text = raw.lower().strip()

    positive = ['yes', 'true', 'y', '1', 'correct', 'affirmative',
                'present', 'visible', 'exists', 'found', 'detected']

    negative = ['no', 'false', 'n', '0', 'incorrect', 'negative',
                'absent', 'not visible', 'not present', 'none',
                'not found', 'not detected', 'cannot', "don't", "doesn't"]

    for neg in negative:
        if neg in text:
            return "0"

    for pos in positive:
        if pos in text:
            return "1"

    return "NA"


def parse_text(raw: str) -> str:
    """Return the raw response with minimal cleaning."""
    text = raw.strip()
    text = ' '.join(text.split())
    return text if text else "NA"


def parse_response(raw: str, task_type: str) -> str:
    """Parse model response based on task type."""
    if task_type == "numeric":
        return parse_numeric(raw)
    elif task_type == "category":
        return parse_category(raw)
    elif task_type == "boolean":
        return parse_boolean(raw)
    elif task_type == "text":
        return parse_text(raw)
    else:
        return parse_text(raw)


# ------------------------------------------------
# Advanced Reasoning Parser (NEW in v2.2)
# ------------------------------------------------
def parse_advanced_reasoning_response(raw: str, task_type: str) -> dict:
    """
    Parse response from advanced reasoning mode.
    Extracts both the reasoning and the final answer.

    Returns:
        dict with keys: answer, reasoning, raw
    """
    raw_stripped = raw.strip()

    # Try to find ANSWER: in the last few lines
    lines = raw_stripped.split('\n')
    answer = "NA"
    reasoning = raw_stripped

    # Search from the end for ANSWER:
    for i in range(len(lines) - 1, max(len(lines) - 5, -1), -1):
        line = lines[i].strip()
        # Case-insensitive search for ANSWER:
        match = re.search(r'answer\s*:\s*(.+)', line, re.IGNORECASE)
        if match:
            answer_text = match.group(1).strip()
            # Parse the answer based on task type
            answer = parse_response(answer_text, task_type)
            # Everything before this line is reasoning
            reasoning = '\n'.join(lines[:i]).strip()
            break

    # If no ANSWER: found, fall back to standard parsing on the whole response
    if answer == "NA":
        answer = parse_response(raw_stripped, task_type)
        reasoning = raw_stripped

    return {
        "answer": answer,
        "reasoning": reasoning,
        "raw": raw_stripped
    }


# ------------------------------------------------
# Consensus Functions (UPDATED in v2.2.1 - Better NA handling)
# ------------------------------------------------
def compute_consensus(parsed_values: list, task_type: str, numeric_tolerance: float = 0.0) -> dict:
    """
    Compute consensus from multiple parsed values using majority voting.

    UPDATED in v2.2.1: NA values are now properly excluded from consensus.
    When valid values exist, NA is never selected as the final answer.

    Args:
        parsed_values: List of parsed values from multiple runs
        task_type: Type of task (numeric, category, boolean)
        numeric_tolerance: For numeric tasks, values within this % are considered equal

    Returns:
        dict with keys: final_value, consensus_reached, agreement_ratio, all_values
    """
    n_runs = len(parsed_values)

    if n_runs == 0:
        return {
            "final_value": "NA",
            "consensus_reached": False,
            "agreement_ratio": 0.0,
            "all_values": []
        }

    # ============================================
    # FIX in v2.2.1: Properly filter out NA values
    # NA can appear as "NA", "na", "N/A", etc.
    # ============================================
    def is_na_value(v):
        """Check if a value represents NA/missing."""
        if v is None:
            return True
        v_str = str(v).strip().upper()
        return v_str in ("NA", "N/A", "NAN", "NONE", "NULL", "")

    valid_values = [v for v in parsed_values if not is_na_value(v)]
    na_count = n_runs - len(valid_values)

    # If all values are NA, return NA
    if len(valid_values) == 0:
        return {
            "final_value": "NA",
            "consensus_reached": False,
            "agreement_ratio": 0.0,
            "all_values": parsed_values
        }

    # ============================================
    # Compute consensus only on valid (non-NA) values
    # ============================================
    if task_type == "numeric" and numeric_tolerance > 0:
        # For numeric with tolerance: group values that are within tolerance
        try:
            numeric_vals = [float(v) for v in valid_values]
            groups = []
            used = [False] * len(numeric_vals)

            for i, val in enumerate(numeric_vals):
                if used[i]:
                    continue
                group = [val]
                used[i] = True
                for j, other_val in enumerate(numeric_vals):
                    if not used[j]:
                        if val != 0:
                            diff_pct = abs(other_val - val) / abs(val)
                        else:
                            diff_pct = abs(other_val - val)
                        if diff_pct <= numeric_tolerance:
                            group.append(other_val)
                            used[j] = True
                groups.append(group)

            largest_group = max(groups, key=len)
            final_value = str(round(sum(largest_group) / len(largest_group), 2))
            agreement_count = len(largest_group)

        except (ValueError, ZeroDivisionError):
            counter = Counter(valid_values)
            final_value, agreement_count = counter.most_common(1)[0]
    else:
        # Standard majority voting (exact match) on valid values only
        counter = Counter(valid_values)
        final_value, agreement_count = counter.most_common(1)[0]

    # ============================================
    # Calculate agreement ratio based on ALL runs (including NA)
    # This gives a true picture of reliability
    # ============================================
    agreement_ratio = agreement_count / n_runs

    # Consensus is reached if more than half of ALL runs agree on the same value
    # (NA runs count against consensus)
    consensus_reached = agreement_ratio > 0.5

    return {
        "final_value": final_value,
        "consensus_reached": consensus_reached,
        "agreement_ratio": round(agreement_ratio, 2),
        "all_values": parsed_values
    }


# ------------------------------------------------
# Reproducibility
# ------------------------------------------------
def set_seed(seed: int | None):
    if seed is None:
        return

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------
# Helper: safe device placement
# ------------------------------------------------
def move_inputs_to_model_if_needed(inputs: dict):
    global MODEL_MAIN_DEVICE

    if "MODEL_MAIN_DEVICE" not in globals():
        MODEL_MAIN_DEVICE = None if hasattr(model, "hf_device_map") else model.device

    if MODEL_MAIN_DEVICE is None:
        return inputs

    device = MODEL_MAIN_DEVICE
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}


# Stores the actual number of generated tokens from the last run_inference call
_last_generated_tokens = 0

# ------------------------------------------------
# Forward pass - Returns RAW response
# UPDATED in v2.2: accepts optional max_tokens_override
# ------------------------------------------------
def run_inference(image_file, prompt, max_tokens_override=None) -> str:
    """
    Sends an image + prompt to the currently loaded VLM.
    Returns the RAW text response (parsing happens separately).

    Args:
        image_file: Path to image or URL
        prompt: The full prompt text
        max_tokens_override: If provided, overrides the global max_new_tokens setting
    """
    global _last_generated_tokens
    # Use override if provided, otherwise use global setting
    tokens_to_use = max_tokens_override if max_tokens_override is not None else max_new_tokens

    image = (
        Image.open(BytesIO(requests.get(image_file).content)).convert("RGB")
        if isinstance(image_file, str) and image_file.startswith("http")
        else Image.open(image_file).convert("RGB")
    )

    if VLM_BACKEND == "llava":
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        prompt_string = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False,
        )

        inputs = processor(
            images=image,
            text=prompt_string,
            return_tensors="pt",
        )

        inputs = move_inputs_to_model_if_needed(inputs)

        output = model.generate(
            **inputs,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=tokens_to_use,
        )

        _last_generated_tokens = len(output[0]) - len(inputs["input_ids"][0])

        raw = processor.decode(output[0], skip_special_tokens=True).strip()

        # LLaVA response cleaning for different base LLMs
        if "[/INST]" in raw:
            raw = raw.split("[/INST]")[-1].strip()

        if "ASSISTANT:" in raw:
            raw = raw.split("ASSISTANT:")[-1].strip()
        elif "Assistant:" in raw:
            raw = raw.split("Assistant:")[-1].strip()

        if "assistant" in raw.lower() and "user" in raw.lower():
            raw_lower = raw.lower()
            idx = raw_lower.rfind("assistant")
            if idx != -1:
                after_assistant = raw[idx + len("assistant"):].lstrip()
                if after_assistant:
                    raw = after_assistant

        return raw

    if VLM_BACKEND == "qwen":
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        inputs = move_inputs_to_model_if_needed(inputs)

        gen_cfg = GenerationConfig.from_model_config(model.config)
        gen_cfg.do_sample = bool(do_sample)
        gen_cfg.max_new_tokens = int(tokens_to_use)

        if gen_cfg.do_sample:
            gen_cfg.temperature = float(temperature)
            gen_cfg.top_p = float(top_p)

        generated_ids = model.generate(
            **inputs,
            generation_config=gen_cfg,
        )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
        ]

        _last_generated_tokens = len(generated_ids_trimmed[0])

        raw = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0].strip()

        return raw

    raise ValueError(f"Unknown VLM_BACKEND='{VLM_BACKEND}'. Expected 'llava' or 'qwen'.")


def caption_image(image_file, prompt, task_type="numeric") -> str:
    """Main entry point: run inference and parse based on task type."""
    raw_response = run_inference(image_file, prompt)
    return parse_response(raw_response, task_type)


# ------------------------------------------------
# UI Widgets
# ------------------------------------------------
folder_widget = widgets.Text(
    description="Image folder",
    placeholder="Folder name in MyDrive/",
    layout=widgets.Layout(width="60%"),
)

role_box = widgets.Textarea(
    description="Role (global)",
    placeholder="This part will be prepended to ALL tasks",
    layout=widgets.Layout(width="100%", height="90px"),
)

n_tasks_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description="Nb tasks",
)

tasks_container = widgets.VBox()

do_sample_widget = widgets.Checkbox(
    value=True,
    description="Enable sampling",
)

temperature_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Temperature",
)

top_p_widget = widgets.FloatSlider(
    value=0.9,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Top-p",
)

max_tokens_widget = widgets.IntSlider(
    value=50,
    min=1,
    max=1500,
    step=1,
    description="Max tokens",
)

display_images_widget = widgets.Checkbox(
    value=True,
    description="Display images during scoring",
)

fixed_seed_widget = widgets.Checkbox(
    value=False,
    description="Use fixed seed",
)

seed_value_widget = widgets.IntText(
    value=42,
    description="Seed value",
    disabled=True,
)

# ------------------------------------------------
# Seed toggle observer
# ------------------------------------------------
def on_fixed_seed_change(change):
    if change["name"] == "value":
        seed_value_widget.disabled = not change["new"]

fixed_seed_widget.observe(on_fixed_seed_change)

# ------------------------------------------------
# Qwen-specific pixel settings
# ------------------------------------------------
DEFAULT_MIN_PIXELS = 256 * 28 * 28
DEFAULT_MAX_PIXELS = 640 * 28 * 28

qwen_settings_label = widgets.HTML(
    value="<b>🔧 Qwen-specific settings</b> <small>(affects VRAM usage vs image quality)</small>"
)

min_pixels_widget = widgets.IntText(
    value=DEFAULT_MIN_PIXELS,
    description="min_pixels",
    layout=widgets.Layout(width="50%"),
)

max_pixels_widget = widgets.IntText(
    value=DEFAULT_MAX_PIXELS,
    description="max_pixels",
    layout=widgets.Layout(width="50%"),
)

qwen_pixels_help = widgets.HTML(
    value="<small><i>Lower values = less VRAM, faster inference, lower quality. "
          "Formula: N × 28 × 28. Common values: 256×28×28=200704 (min), 640×28×28=501760 (default max), 1280×28×28=1003520 (high quality)</i></small>"
)

qwen_settings_box = widgets.VBox(
    [qwen_settings_label, min_pixels_widget, max_pixels_widget, qwen_pixels_help],
    layout=widgets.Layout(
        border="1px solid #4a9eff",
        padding="10px",
        margin="10px 0px",
        display="none",
    )
)

def update_qwen_settings_visibility():
    """Show Qwen settings only if Qwen backend is loaded."""
    if "VLM_BACKEND" in globals() and VLM_BACKEND == "qwen":
        qwen_settings_box.layout.display = "block"
        if "QWEN_MIN_PIXELS" in globals():
            min_pixels_widget.value = QWEN_MIN_PIXELS
        if "QWEN_MAX_PIXELS" in globals():
            max_pixels_widget.value = QWEN_MAX_PIXELS
    else:
        qwen_settings_box.layout.display = "none"

update_qwen_settings_visibility()


apply_button = widgets.Button(
    description="Apply paths + tasks + settings",
    button_style="success",
)

status_out = widgets.Output()

# ------------------------------------------------
# Dynamic task widgets builder (UPDATED for v2.2 - Advanced Reasoning)
# ------------------------------------------------
TASK_WIDGETS = []

def build_task_widgets(n: int):
    global TASK_WIDGETS

    TASK_WIDGETS = []
    blocks = []

    for i in range(1, n + 1):
        task_type = widgets.Dropdown(
            options=[
                ("Numeric (counts, scores)", "numeric"),
                ("Category (classifications)", "category"),
                ("Boolean (yes/no)", "boolean"),
                ("Text (free-form)", "text"),
            ],
            value="numeric",
            description=f"Type {i}",
            layout=widgets.Layout(width="60%"),
        )

        col_title = widgets.Text(
            description=f"Col {i}",
            value=(f"task_{i}" if i > 1 else "task_1"),
            layout=widgets.Layout(width="60%"),
        )

        theory = widgets.Textarea(
            description=f"Theory {i}",
            placeholder=f"Scoring rules / theoretical guidance (task {i})",
            layout=widgets.Layout(width="100%", height="110px"),
        )

        task = widgets.Textarea(
            description=f"Task {i}",
            placeholder=f"What the model must do (task {i})",
            layout=widgets.Layout(width="100%", height="80px"),
        )

        fmt = widgets.Textarea(
            description=f"Format {i}",
            placeholder=f"Expected output format (task {i})",
            layout=widgets.Layout(width="100%", height="60px"),
        )

        help_text = widgets.HTML(
            value="<small><i>Numeric: extracts numbers | Category: returns classification text | Boolean: normalizes to 1/0 | Text: raw response</i></small>"
        )

        # ============================================
        # NEW in v2.2: Advanced Reasoning checkbox
        # ============================================
        advanced_reasoning_checkbox = widgets.Checkbox(
            value=False,
            description="Enable advanced reasoning (chain-of-thought)",
            layout=widgets.Layout(width="auto"),
        )

        advanced_reasoning_help = widgets.HTML(
            value="<small><i>🧠 Model will describe observations and reasoning before answering. "
                  f"Max tokens automatically set to {ADVANCED_REASONING_MAX_TOKENS}. "
                  "Creates additional columns for reasoning text and truncation warnings.</i></small>",
            layout=widgets.Layout(display="none"),
        )

        advanced_reasoning_format_preview = widgets.HTML(
            value="",
            layout=widgets.Layout(display="none"),
        )

        # ============================================
        # Consensus widgets (from v2.1)
        # ============================================
        consensus_checkbox = widgets.Checkbox(
            value=False,
            description="Enable consensus validation",
            layout=widgets.Layout(width="auto"),
        )

        consensus_runs = widgets.Dropdown(
            options=[("2 runs", 2), ("3 runs", 3), ("4 runs", 4), ("5 runs", 5)],
            value=2,
            description="Runs",
            disabled=True,
            layout=widgets.Layout(width="30%"),
        )

        numeric_tolerance = widgets.FloatText(
            value=0.0,
            description="Tolerance %",
            disabled=False,
            layout=widgets.Layout(width="30%"),
        )

        consensus_help = widgets.HTML(
            value="<small><i>⚠️ Consensus multiplies API calls! Tolerance: for numeric, values within X% are considered equal (0 = exact match)</i></small>",
            layout=widgets.Layout(display="none"),
        )

        consensus_cost_warning = widgets.HTML(
            value="",
            layout=widgets.Layout(display="none"),
        )

        consensus_options_box = widgets.HBox(
            [consensus_runs, numeric_tolerance],
            layout=widgets.Layout(display="none"),
        )

        # ============================================
        # Observers for Advanced Reasoning (NEW in v2.2)
        # ============================================
        def make_advanced_reasoning_observer(checkbox, help_widget, preview_widget, format_widget, task_type_widget):
            def observer(change):
                if change["name"] == "value":
                    is_enabled = change["new"]
                    current_type = task_type_widget.value

                    help_widget.layout.display = "block" if is_enabled else "none"
                    preview_widget.layout.display = "block" if is_enabled else "none"

                    # Disable format field when advanced reasoning is ON
                    format_widget.disabled = is_enabled

                    if is_enabled and current_type != "text":
                        format_text = ADVANCED_REASONING_FORMATS.get(current_type, "")
                        preview_widget.value = f"<small><b>Format (auto):</b><br><code style='white-space: pre-wrap;'>{format_text}</code></small>"
                        format_widget.value = ""  # Clear user format
                    else:
                        preview_widget.value = ""
            return observer

        def make_type_observer_v2(tolerance_widget, adv_checkbox, adv_help, adv_preview, format_widget):
            def observer(change):
                if change["name"] == "value":
                    new_type = change["new"]

                    # Enable/disable tolerance for numeric
                    tolerance_widget.disabled = new_type != "numeric"

                    # Disable advanced reasoning for text type
                    if new_type == "text":
                        adv_checkbox.value = False
                        adv_checkbox.disabled = True
                        adv_help.layout.display = "none"
                        adv_preview.layout.display = "none"
                        format_widget.disabled = False
                    else:
                        adv_checkbox.disabled = False
                        # Update preview if advanced reasoning is enabled
                        if adv_checkbox.value:
                            format_text = ADVANCED_REASONING_FORMATS.get(new_type, "")
                            adv_preview.value = f"<small><b>Format (auto):</b><br><code style='white-space: pre-wrap;'>{format_text}</code></small>"
            return observer

        # Consensus observers (from v2.1)
        def make_consensus_observer(checkbox, options_box, help_widget, cost_widget, runs_widget, task_type_widget):
            def observer(change):
                if change["name"] == "value":
                    is_enabled = change["new"]
                    options_box.layout.display = "flex" if is_enabled else "none"
                    help_widget.layout.display = "block" if is_enabled else "none"
                    cost_widget.layout.display = "block" if is_enabled else "none"
                    runs_widget.disabled = not is_enabled

                    if is_enabled:
                        n_runs = runs_widget.value
                        cost_widget.value = f"<small style='color: #ff6600;'><b>⚠️ Cost: {n_runs}x API calls per image for this task</b></small>"
                    else:
                        cost_widget.value = ""
            return observer

        def make_runs_observer(cost_widget, checkbox):
            def observer(change):
                if change["name"] == "value" and checkbox.value:
                    n_runs = change["new"]
                    cost_widget.value = f"<small style='color: #ff6600;'><b>⚠️ Cost: {n_runs}x API calls per image for this task</b></small>"
            return observer

        # Attach observers
        advanced_reasoning_checkbox.observe(
            make_advanced_reasoning_observer(advanced_reasoning_checkbox, advanced_reasoning_help,
                                            advanced_reasoning_format_preview, fmt, task_type)
        )

        task_type.observe(
            make_type_observer_v2(numeric_tolerance, advanced_reasoning_checkbox,
                                 advanced_reasoning_help, advanced_reasoning_format_preview, fmt)
        )

        consensus_checkbox.observe(
            make_consensus_observer(consensus_checkbox, consensus_options_box, consensus_help,
                                   consensus_cost_warning, consensus_runs, task_type)
        )
        consensus_runs.observe(make_runs_observer(consensus_cost_warning, consensus_checkbox))

        TASK_WIDGETS.append({
            "task_type": task_type,
            "col_title": col_title,
            "theory": theory,
            "task": task,
            "format": fmt,
            # Advanced reasoning (NEW in v2.2)
            "advanced_reasoning": advanced_reasoning_checkbox,
            # Consensus widgets
            "consensus_enabled": consensus_checkbox,
            "consensus_runs": consensus_runs,
            "numeric_tolerance": numeric_tolerance,
        })

        # Advanced Reasoning section (NEW in v2.2)
        advanced_reasoning_section = widgets.VBox(
            [
                widgets.HTML("<hr style='margin: 5px 0; border-color: #ddd;'>"),
                widgets.HTML("<b style='font-size: 0.9em;'>🧠 Advanced Reasoning</b>"),
                advanced_reasoning_checkbox,
                advanced_reasoning_help,
                advanced_reasoning_format_preview,
            ],
            layout=widgets.Layout(margin="5px 0 0 0"),
        )

        # Consensus section
        consensus_section = widgets.VBox(
            [
                widgets.HTML("<hr style='margin: 5px 0; border-color: #ddd;'>"),
                widgets.HTML("<b style='font-size: 0.9em;'>🔄 Consensus Validation</b>"),
                consensus_checkbox,
                consensus_options_box,
                consensus_help,
                consensus_cost_warning,
            ],
            layout=widgets.Layout(margin="5px 0 0 0"),
        )

        block = widgets.VBox(
            [
                widgets.HTML(f"<b>Task #{i}</b>"),
                task_type,
                col_title,
                task,
                theory,
                fmt,
                help_text,
                advanced_reasoning_section,
                consensus_section,
            ],
            layout=widgets.Layout(
                border="1px solid #ddd",
                padding="10px",
                margin="6px 0px 10px 0px",
            ),
        )
        blocks.append(block)

    tasks_container.children = tuple(blocks)


build_task_widgets(n_tasks_widget.value)

def on_n_tasks_change(change):
    if change["name"] == "value":
        build_task_widgets(int(change["new"]))

n_tasks_widget.observe(on_n_tasks_change)


# ------------------------------------------------
# Apply callback (UPDATED for v2.2 - Advanced Reasoning)
# ------------------------------------------------
def apply_settings(b):
    global image_folder, output_path
    global do_sample, temperature, top_p, max_new_tokens
    global display_images, generation_seed
    global TASK_SPECS
    global processor
    global QWEN_MIN_PIXELS, QWEN_MAX_PIXELS

    root_path = "/content/drive/MyDrive"
    folder_name = folder_widget.value.strip()

    image_folder = os.path.join(root_path, folder_name)

    out_name = "Score_Analysis_QwenVL.csv" if VLM_BACKEND == "qwen" else "Score_Analysis_LLaVA.csv"
    output_path = os.path.join(image_folder, out_name)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    do_sample = do_sample_widget.value
    temperature = float(temperature_widget.value)
    top_p = float(top_p_widget.value)
    max_new_tokens = int(max_tokens_widget.value)
    display_images = bool(display_images_widget.value)

    if fixed_seed_widget.value:
        generation_seed = int(seed_value_widget.value)
        set_seed(generation_seed)
    else:
        generation_seed = None

    # Handle Qwen pixel settings
    qwen_pixels_changed = False
    if VLM_BACKEND == "qwen":
        new_min = int(min_pixels_widget.value)
        new_max = int(max_pixels_widget.value)

        if "QWEN_MIN_PIXELS" in globals() and "QWEN_MAX_PIXELS" in globals():
            if new_min != QWEN_MIN_PIXELS or new_max != QWEN_MAX_PIXELS:
                qwen_pixels_changed = True
                print(f"🔄 Reloading Qwen processor with new pixel settings...")
                print(f"   min_pixels: {QWEN_MIN_PIXELS} → {new_min}")
                print(f"   max_pixels: {QWEN_MAX_PIXELS} → {new_max}")

                auth_kwargs = {}
                if "HF_TOKEN" in globals() and HF_TOKEN:
                    auth_kwargs["token"] = HF_TOKEN

                processor = AutoProcessor.from_pretrained(
                    model_id,
                    min_pixels=new_min,
                    max_pixels=new_max,
                    **auth_kwargs
                )

                QWEN_MIN_PIXELS = new_min
                QWEN_MAX_PIXELS = new_max
                print("✅ Processor reloaded.")

    role_txt = role_box.value.strip()

    TASK_SPECS = []
    seen_cols = set()
    total_consensus_cost = 0
    any_advanced_reasoning = False

    for i, tw in enumerate(TASK_WIDGETS, start=1):
        task_type = tw["task_type"].value
        col = tw["col_title"].value.strip()
        th = tw["theory"].value.strip()
        tk = tw["task"].value.strip()
        fm = tw["format"].value.strip()

        # Advanced reasoning (NEW in v2.2)
        advanced_reasoning = tw["advanced_reasoning"].value

        # Advanced reasoning not available for text type
        if task_type == "text" and advanced_reasoning:
            advanced_reasoning = False

        if advanced_reasoning:
            any_advanced_reasoning = True
            # Override format with advanced reasoning format
            fm = ADVANCED_REASONING_FORMATS.get(task_type, fm)

        # Consensus settings
        consensus_enabled = tw["consensus_enabled"].value
        consensus_runs = tw["consensus_runs"].value if consensus_enabled else 1
        numeric_tolerance = tw["numeric_tolerance"].value if task_type == "numeric" else 0.0

        # Consensus only available for numeric, category, boolean (not text)
        if task_type == "text" and consensus_enabled:
            consensus_enabled = False
            consensus_runs = 1

        if consensus_enabled:
            total_consensus_cost += consensus_runs
        else:
            total_consensus_cost += 1

        if not col:
            col = f"task_{i}"

        if col in seen_cols:
            col = f"{col}_{i}"
        seen_cols.add(col)

        full_prompt = f"""
{role_txt}

{tk}

{th}

{fm}
""".strip()

        TASK_SPECS.append({
            "column": col,
            "prompt": full_prompt,
            "task_type": task_type,
            # Advanced reasoning (NEW in v2.2)
            "advanced_reasoning": advanced_reasoning,
            # Consensus config
            "consensus_enabled": consensus_enabled,
            "consensus_runs": consensus_runs,
            "numeric_tolerance": numeric_tolerance / 100.0,
        })

    status_out.clear_output(wait=True)
    with status_out:
        print("✅ Settings applied\n")

        print("----- BACKEND -----")
        print("🧠 VLM backend:", VLM_BACKEND)

        if "DEVICE_MAP_MODE" in globals():
            print("🧩 Placement:", DEVICE_MAP_MODE)

        if VLM_BACKEND == "qwen":
            print(f"📐 Qwen min_pixels: {min_pixels_widget.value}")
            print(f"📐 Qwen max_pixels: {max_pixels_widget.value}")
            if qwen_pixels_changed:
                print("   (processor reloaded with new settings)")

        print("\n----- PATHS -----")
        print("📁 Image folder:", image_folder)
        print("💾 Output file:", output_path)

        print("\n----- TASKS -----")
        print(f"Nb tasks: {len(TASK_SPECS)}")
        for j, spec in enumerate(TASK_SPECS, start=1):
            flags = []
            if spec["advanced_reasoning"]:
                flags.append(f"🧠 reasoning")
            if spec["consensus_enabled"]:
                tol_str = f", tol={spec['numeric_tolerance']*100:.1f}%" if spec['task_type'] == 'numeric' and spec['numeric_tolerance'] > 0 else ""
                flags.append(f"🔄 consensus={spec['consensus_runs']}x{tol_str}")
            flags_str = " | " + " | ".join(flags) if flags else ""
            print(f"  {j}. column='{spec['column']}' | type={spec['task_type']}{flags_str}")

        print("\n----- GENERATION -----")
        print(f"do_sample      = {do_sample}")
        print(f"temperature    = {temperature}")
        print(f"top_p          = {top_p}")
        print(f"max_new_tokens = {max_new_tokens}")
        if any_advanced_reasoning:
            print(f"   ⚠️  Advanced reasoning tasks will use max_tokens={ADVANCED_REASONING_MAX_TOKENS}")
        print(f"display_images = {display_images}")
        print(f"fixed_seed     = {generation_seed}")

        # Cost warning
        print("\n----- COST ESTIMATE -----")
        print(f"📊 Total API calls per image: {total_consensus_cost}")
        if total_consensus_cost > len(TASK_SPECS):
            print(f"⚠️  Consensus enabled: {total_consensus_cost - len(TASK_SPECS)} extra calls per image")

apply_button.on_click(apply_settings)


# ------------------------------------------------
# Display UI
# ------------------------------------------------
display(
    folder_widget,
    role_box,
    n_tasks_widget,
    tasks_container,
    qwen_settings_box,
    display_images_widget,
    do_sample_widget,
    temperature_widget,
    top_p_widget,
    max_tokens_widget,
    fixed_seed_widget,
    seed_value_widget,
    apply_button,
    status_out,
)

update_qwen_settings_visibility()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text(value='', description='Image folder', layout=Layout(width='60%'), placeholder='Folder name in MyDrive/')

Textarea(value='', description='Role (global)', layout=Layout(height='90px', width='100%'), placeholder='This …

IntSlider(value=1, description='Nb tasks', max=10, min=1)

Checkbox(value=True, description='Display images during scoring')

...

IntSlider(value=50, description='Max tokens', max=500, min=1)

Checkbox(value=False, description='Use fixed seed')

IntText(value=42, description='Seed value', disabled=True)

Button(button_style='success', description='Apply paths + tasks + settings', style=ButtonStyle())

Output()

In [3]:
# ===============================================
# BLOCK 3 – Run Analysis with Task Type Support
#           + CONSENSUS VALIDATION + ADVANCED REASONING (v2.2.2)
#           FIX: Removed truncation of raw/reasoning columns
# ===============================================

import time
import json
import pandas as pd
from PIL import Image
from IPython.display import display

# ------------------------------------------------
# Safety checks
# ------------------------------------------------
if "TASK_SPECS" not in globals() or not isinstance(TASK_SPECS, list) or len(TASK_SPECS) == 0:
    raise RuntimeError("TASK_SPECS is missing/empty. Run Block 2 and click 'Apply paths + tasks + settings' first.")

if "image_folder" not in globals() or "output_path" not in globals():
    raise RuntimeError("image_folder/output_path not set. Run Block 2 and click 'Apply paths + tasks + settings' first.")

for spec in TASK_SPECS:
    if "task_type" not in spec:
        raise RuntimeError("TASK_SPECS missing 'task_type'. Please re-run Block 2 (v2.2) and click Apply.")

# ------------------------------------------------
# Build column headers (UPDATED for v2.2 - Advanced Reasoning)
# ------------------------------------------------
task_columns = [spec["column"] for spec in TASK_SPECS]

# For consensus tasks, we add extra columns
consensus_columns = []
# For advanced reasoning tasks, we add reasoning + truncated columns
reasoning_columns = []
truncated_columns = []
raw_columns = []

for spec in TASK_SPECS:
    col = spec["column"]
    raw_columns.append(f"{col}_raw")

    if spec.get("consensus_enabled", False):
        consensus_columns.append(f"{col}_consensus")
        consensus_columns.append(f"{col}_agreement")
        consensus_columns.append(f"{col}_runs")

    # Add truncation column for ALL tasks
    truncated_columns.append(f"{col}_truncated")

    # Add reasoning column only for advanced reasoning tasks
    if spec.get("advanced_reasoning", False):
        reasoning_columns.append(f"{col}_reasoning")

header = ["image_name"] + task_columns + reasoning_columns + truncated_columns + consensus_columns + raw_columns

start_time = time.time()

# ------------------------------------------------
# Load existing CSV
# ------------------------------------------------
if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
    df = pd.read_csv(output_path, dtype=str)
    print(f"📂 Resuming previous run – {len(df)} rows already in CSV.")
else:
    df = pd.DataFrame(columns=["image_name"])
    print("🆕 Starting fresh – no previous CSV found.")

if "image_name" not in df.columns:
    raise RuntimeError("Existing CSV has no 'image_name' column. Cannot resume safely.")

df["image_name"] = df["image_name"].astype(str)

# ------------------------------------------------
# Schema upgrade
# ------------------------------------------------
missing_cols = [c for c in header if c not in df.columns]
if len(missing_cols) > 0:
    print("🧩 Existing CSV is missing columns; upgrading schema:")
    for c in missing_cols:
        print(f"  + adding column: {c}")
        df[c] = "NA"
    print("✅ CSV schema upgraded.\n")

ordered = header + [c for c in df.columns if c not in header]
df = df[ordered]

# ------------------------------------------------
# Build list of images
# ------------------------------------------------
image_files = [
    f for f in sorted(os.listdir(image_folder))
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"📷 Found {len(image_files)} images in folder.")

# Show consensus info
consensus_tasks = [s for s in TASK_SPECS if s.get("consensus_enabled", False)]
if consensus_tasks:
    print(f"🔄 Consensus enabled for {len(consensus_tasks)} task(s):")
    for ct in consensus_tasks:
        print(f"   - {ct['column']}: {ct['consensus_runs']} runs")

# Show advanced reasoning info (NEW in v2.2)
reasoning_tasks = [s for s in TASK_SPECS if s.get("advanced_reasoning", False)]
if reasoning_tasks:
    print(f"🧠 Advanced reasoning enabled for {len(reasoning_tasks)} task(s):")
    for rt in reasoning_tasks:
        print(f"   - {rt['column']} (max_tokens={ADVANCED_REASONING_MAX_TOKENS})")

print()

idx_map = {name: i for i, name in enumerate(df["image_name"].tolist())}


# ------------------------------------------------
# Truncation detection for advanced reasoning
# ------------------------------------------------
def check_truncation(max_tokens: int) -> tuple:
    """
    Detect if the last model response was truncated by the token limit.
    Uses _last_generated_tokens set by run_inference (exact count from model output).
    
    Returns (is_truncated: bool, token_count: int)
    """
    return _last_generated_tokens >= max_tokens, _last_generated_tokens

# ------------------------------------------------
# Main processing loop (UPDATED for v2.2 - Advanced Reasoning)
# ------------------------------------------------
processed_count = 0
skipped_count = 0
total_api_calls = 0

for fname in image_files:
    image_path = os.path.join(image_folder, fname)
    print(f"\n📍 Processing: {fname}")

    if display_images:
        try:
            display(Image.open(image_path))
        except Exception as e:
            print(f"⚠️ Could not display image: {e}")

    if fname not in idx_map:
        new_row = {c: "NA" for c in df.columns}
        new_row["image_name"] = fname
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        idx_map[fname] = len(df) - 1

    row_i = idx_map[fname]
    image_processed = False

    for spec in TASK_SPECS:
        col = spec["column"]
        task_prompt = spec["prompt"]
        task_type = spec["task_type"]
        raw_col = f"{col}_raw"
        reasoning_col = f"{col}_reasoning"

        # Advanced reasoning settings (NEW in v2.2)
        advanced_reasoning = spec.get("advanced_reasoning", False)

        # Consensus settings
        consensus_enabled = spec.get("consensus_enabled", False)
        consensus_runs = spec.get("consensus_runs", 1)
        numeric_tolerance = spec.get("numeric_tolerance", 0.0)

        current = df.at[row_i, col] if col in df.columns else "NA"
        current = "" if pd.isna(current) else str(current).strip()

        needs_compute = (current == "") or (current.upper() == "NA") or current.startswith("ERROR:")

        if not needs_compute:
            print(f"  ⭐ {col}: already set ({current})")
            skipped_count += 1
            continue

        try:
            if consensus_enabled and task_type != "text":
                # ============================================
                # CONSENSUS MODE: Run multiple times
                # ============================================
                print(f"  🔄 {col}: Running {consensus_runs}x for consensus...")

                raw_responses = []
                parsed_values = []
                reasoning_texts = []

                for run_i in range(consensus_runs):
                    if advanced_reasoning:
                        raw_response = run_inference(image_path, task_prompt,
                                                    max_tokens_override=ADVANCED_REASONING_MAX_TOKENS)
                        parsed_result = parse_advanced_reasoning_response(raw_response, task_type)
                        parsed_value = parsed_result["answer"]
                        reasoning_texts.append(parsed_result["reasoning"])
                    else:
                        raw_response = run_inference(image_path, task_prompt)
                        parsed_value = parse_response(raw_response, task_type)

                    raw_responses.append(raw_response)
                    parsed_values.append(parsed_value)
                    total_api_calls += 1

                    print(f"      Run {run_i+1}: {parsed_value}")

                # Compute consensus
                consensus_result = compute_consensus(
                    parsed_values,
                    task_type,
                    numeric_tolerance
                )

                # Store results
                df.at[row_i, col] = consensus_result["final_value"]
                df.at[row_i, f"{col}_consensus"] = "YES" if consensus_result["consensus_reached"] else "NO"
                df.at[row_i, f"{col}_agreement"] = str(consensus_result["agreement_ratio"])
                df.at[row_i, f"{col}_runs"] = json.dumps(consensus_result["all_values"])
                df.at[row_i, raw_col] = raw_responses[0]

                # Store first reasoning if advanced reasoning enabled
                if advanced_reasoning and reasoning_texts:
                    df.at[row_i, reasoning_col] = reasoning_texts[0]

                # Truncation detection (applies to ALL consensus tasks)
                truncated_col = f"{col}_truncated"
                effective_max = ADVANCED_REASONING_MAX_TOKENS if advanced_reasoning else max_new_tokens
                is_truncated, token_count = check_truncation(effective_max)
                df.at[row_i, truncated_col] = "YES" if is_truncated else "NO"
                if is_truncated:
                    print(f"  🚨 {col}: TRUNCATION DETECTED — response used {token_count}/{effective_max} tokens. Increase max_tokens!")

                consensus_icon = "✅" if consensus_result["consensus_reached"] else "⚠️"
                print(f"  {consensus_icon} {col} [{task_type}]: {consensus_result['final_value']} (agreement: {consensus_result['agreement_ratio']:.0%})")

            elif advanced_reasoning and task_type != "text":
                # ============================================
                # ADVANCED REASONING MODE
                # ============================================
                raw_response = run_inference(image_path, task_prompt,
                                            max_tokens_override=ADVANCED_REASONING_MAX_TOKENS)
                parsed_result = parse_advanced_reasoning_response(raw_response, task_type)
                total_api_calls += 1

                df.at[row_i, col] = str(parsed_result["answer"]).strip()
                df.at[row_i, reasoning_col] = str(parsed_result["reasoning"]).strip()
                df.at[row_i, raw_col] = str(raw_response).strip()

                # Truncation detection
                truncated_col = f"{col}_truncated"
                is_truncated, token_count = check_truncation(ADVANCED_REASONING_MAX_TOKENS)
                df.at[row_i, truncated_col] = "YES" if is_truncated else "NO"

                print(f"  🧠 {col} [{task_type}]: {df.at[row_i, col]}")
                if is_truncated:
                    print(f"  🚨 {col}: TRUNCATION DETECTED — response used {token_count}/{ADVANCED_REASONING_MAX_TOKENS} tokens. Increase max_tokens!")
                else:
                    reasoning_preview = parsed_result["reasoning"][:100] + "..." if len(parsed_result["reasoning"]) > 100 else parsed_result["reasoning"]
                    print(f"      (reasoning: {reasoning_preview})")

            else:
                # ============================================
                # STANDARD MODE: Single run
                # ============================================
                raw_response = run_inference(image_path, task_prompt)
                parsed_result = parse_response(raw_response, task_type)
                total_api_calls += 1

                df.at[row_i, col] = str(parsed_result).strip()
                df.at[row_i, raw_col] = str(raw_response).strip()

                # Truncation detection (applies to ALL tasks)
                truncated_col = f"{col}_truncated"
                is_truncated, token_count = check_truncation(max_new_tokens)
                df.at[row_i, truncated_col] = "YES" if is_truncated else "NO"

                print(f"  📝 {col} [{task_type}]: {df.at[row_i, col]}")
                if is_truncated:
                    print(f"  🚨 {col}: TRUNCATION DETECTED — response used {token_count}/{max_new_tokens} tokens. Increase max_tokens!")
                elif task_type != "text":
                    raw_preview = raw_response[:80] + "..." if len(raw_response) > 80 else raw_response
                    print(f"      (raw: {raw_preview})")

            image_processed = True

        except Exception as e:
            err = f"ERROR: {e}"
            df.at[row_i, col] = err
            df.at[row_i, raw_col] = err
            print(f"  ❌ {col}: {err}")

    if image_processed:
        processed_count += 1

    if processed_count > 0 and processed_count % 3 == 0:
        df.to_csv(output_path, index=False)
        print(f"  💾 Checkpoint saved ({processed_count} images processed)")

# ------------------------------------------------
# Final save
# ------------------------------------------------
df.to_csv(output_path, index=False)

elapsed_time = time.time() - start_time

print("\n" + "="*50)
print("✅ Analysis completed.")
print(f"📄 Results saved to: {output_path}")
print(f"🧠 Tasks written: {len(TASK_SPECS)}")
for spec in TASK_SPECS:
    flags = []
    if spec.get("advanced_reasoning"):
        flags.append("🧠 reasoning")
    if spec.get("consensus_enabled"):
        flags.append(f"🔄 consensus={spec['consensus_runs']}x")
    flags_str = f" ({', '.join(flags)})" if flags else ""
    print(f"   - {spec['column']} ({spec['task_type']}){flags_str}")
print(f"📷 Images processed: {processed_count}")
print(f"⭐️ Tasks skipped (already done): {skipped_count}")
print(f"📢 Total API calls: {total_api_calls}")
print(f"⏱️ Total processing time: {elapsed_time:.2f} seconds")
print("="*50)


Output hidden; open in https://colab.research.google.com to view.